# 03 · 特征筛选模块（Selectors）功能演示

逐一运行 22 种特征筛选器，演示中文筛选报告、组合筛选器与报告收集器。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 22 种特征筛选器逐一运行
过滤法 / 包装法 / 嵌入法，统一 fit/transform/get_support_mask/get_selection_report 接口，报告均为中文键。

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression as SkLR
from hscredit.core.selectors import (TypeSelector, RegexSelector, NullSelector, ModeSelector,
    CardinalitySelector, VarianceSelector, CorrSelector, VIFSelector, IVSelector, LiftSelector,
    PSISelector, FeatureImportanceSelector, NullImportanceSelector, RFESelector,
    SequentialFeatureSelector, StepwiseSelector, BorutaSelector, MutualInfoSelector, Chi2Selector,
    FTestSelector, StabilityAwareSelector, ScorecardFeatureSelection, CompositeFeatureSelector)
from hscredit.core.selectors.base import SelectionReportCollector

X = df[NUM_FEATURES].copy()
Xnn = X.fillna(0).clip(lower=0)   # chi2 等要求非负 / 部分 sklearn 模型要求无 NaN

makers = [
    ('类型筛选 TypeSelector', lambda: TypeSelector(dtype_include='number'), X),
    ('正则筛选 RegexSelector', lambda: RegexSelector(pattern='分'), X),
    ('缺失率 NullSelector', lambda: NullSelector(threshold=0.95), X),
    ('众数占比 ModeSelector', lambda: ModeSelector(threshold=0.95), X),
    ('基数 CardinalitySelector', lambda: CardinalitySelector(), X),
    ('方差 VarianceSelector', lambda: VarianceSelector(threshold=0.0), X),
    ('相关性 CorrSelector', lambda: CorrSelector(threshold=0.9), X),
    ('VIF VIFSelector', lambda: VIFSelector(threshold=10.0), X),
    ('IV IVSelector', lambda: IVSelector(threshold=0.02), X),
    ('Lift LiftSelector', lambda: LiftSelector(), X),
    ('PSI PSISelector', lambda: PSISelector(threshold=0.25), X),
    ('重要性 FeatureImportanceSelector', lambda: FeatureImportanceSelector(RandomForestClassifier(n_estimators=30, random_state=0)), Xnn),
    ('零重要性 NullImportanceSelector', lambda: NullImportanceSelector(RandomForestClassifier(n_estimators=30, random_state=0), n_runs=3), Xnn),
    ('RFE RFESelector', lambda: RFESelector(SkLR(max_iter=200), n_features_to_select=5), Xnn),
    ('顺序 SequentialFeatureSelector', lambda: SequentialFeatureSelector(SkLR(max_iter=200), n_features_to_select=4), Xnn),
    ('逐步回归 StepwiseSelector', lambda: StepwiseSelector(), X),
    ('Boruta BorutaSelector', lambda: BorutaSelector(max_iter=20), X),
    ('互信息 MutualInfoSelector', lambda: MutualInfoSelector(), X),
    ('卡方 Chi2Selector', lambda: Chi2Selector(), Xnn),
    ('F检验 FTestSelector', lambda: FTestSelector(), X),
    ('稳定性 StabilityAwareSelector', lambda: StabilityAwareSelector(), X),
    ('评分卡组合 ScorecardFeatureSelection', lambda: ScorecardFeatureSelection(), X),
]
rows = []
for name, make, Xd in makers:
    s = make().fit(Xd, y)
    rep = s.get_selection_report()
    rows.append({'筛选器': name, '输入特征数': rep['输入特征数'], '选中特征数': rep['选中特征数']})
pd.DataFrame(rows)

,筛选器,输入特征数,选中特征数
0,类型筛选 TypeSelector,6,6
1,正则筛选 RegexSelector,6,2
2,缺失率 NullSelector,6,6
3,众数占比 ModeSelector,6,6
4,基数 CardinalitySelector,6,0
5,方差 VarianceSelector,6,6
6,相关性 CorrSelector,6,6
7,VIF VIFSelector,6,6
8,IV IVSelector,6,5
9,Lift LiftSelector,6,3


## 2. IV 筛选报告详情（中文键）

In [3]:
iv_sel = IVSelector(threshold=0.02).fit(X, y)
rep = iv_sel.get_selection_report()
print('筛选方法:', rep['筛选方法'])
print('选中特征:', rep['选中特征'])
iv_sel.get_selection_report_df()

筛选方法: IV值筛选
选中特征: ['珊瑚92', '青云24', '占信V3', '天创小额网贷分', '近六个月非银多头机构数']


,筛选器,筛选方法,阈值,输入特征数,选中特征数,剔除特征数,保留率
0,IVSelector,IV值筛选,0.0200,6,5,1,83.33%


## 3. 组合筛选器 + 报告收集器（统一中文摘要）

In [4]:
comp = CompositeFeatureSelector(selectors=[NullSelector(threshold=0.99), IVSelector(threshold=0.02), CorrSelector(threshold=0.9)])
comp.fit(X, y)
X_selected = comp.transform(X)
print('组合筛选后特征数:', X_selected.shape[1])

collector = SelectionReportCollector()
for sel in [NullSelector(threshold=0.99), IVSelector(threshold=0.02)]:
    sel.fit(X, y); collector.add_report(sel)
collector.get_summary()

组合筛选后特征数: 5


{'流程名称': '特征筛选流程',
 '创建时间': '2026-06-23 05:52:30',
 '筛选轮次': 2,
 '原始特征数': 6,
 '最终特征数': 5,
 '累计剔除特征数': 1,
 '特征保留率': '83.33%',
 '筛选器列表': [{'阶段': '阶段1',
   '筛选器': 'NullSelector',
   '输入': 6,
   '输出': 6,
   '剔除': 0,
   '阈值': 0.99},
  {'阶段': '阶段2', '筛选器': 'IVSelector', '输入': 6, '输出': 5, '剔除': 1, '阈值': 0.02}]}

## 4. 筛选报告导出到 Excel

In [5]:
collector.to_excel(f"{OUT}/03_selectors_report.xlsx")
pd.DataFrame(rows).to_excel(f"{OUT}/03_selectors_overview.xlsx", index=False)
print('已保存筛选报告')

已保存筛选报告
